In [2]:
import json
with open('output/answers.jsonl') as f:
    answers = [json.loads(line) for line in f]


In [3]:
with open('/home/v-jinjzhao/datasets/rec_human_celeb_ready/person_annts_val_ready.json') as f:
    all_annts=json.load(f)

pred_bboxes=[]
gt_bboxes=[]
for idx,annt in enumerate(all_annts):
    pred=answers[idx]
    assert pred['annt_id']==annt['annt_id'], f"{pred['annt_id']}!={annt['annt_id']}"
    pred_bbox=pred['pred_bboxes'][0]
    gt_bbox=annt['bbox']
    if(annt['recognition_category']=='celebrity'):
        x1,y1,x2,y2=gt_bbox
    else:
        x,y,w,h=gt_bbox
        x1,y1,x2,y2=[x,y,x+w,y+h]
    pred_bboxes.append(pred_bbox)
    gt_bboxes.append([x1,y1,x2,y2])



In [4]:
import json
import torch
from tqdm import tqdm
from overlaps import bbox_overlaps
# import average from math
from statistics import mean


def calculate_iou_acc(pred_bboxes,gt_bboxes, thresh=0.5):
    """
    pred_bboxes: [N,4]
    gt_bboxes: [N,4]
    calculate the iou and acc of the pred_bboxes and gt_bboxes,
    if iou(pred_bboxes[i],gt_bboxes[i])>0.5, then acc+=1
    all pred_bboxes_i and gt_bboxes_i are one to one assigned.
    
    """
    iou=bbox_overlaps(pred_bboxes,gt_bboxes,mode='iou', is_aligned=True)
    if(type(thresh) is not list):
        thresh=[thresh]
    accs=dict()
    for t in thresh:
        accs[t]=(iou>t).sum().item()/len(iou)
    return iou,accs




pred_bboxes=torch.tensor(pred_bboxes)
gt_bboxes=torch.tensor(gt_bboxes)
# create a list fron 0.5 to 0.9 with step 0.05
thresh=[i/100 for i in range(50,95,5)]
print(thresh)
iou,acc=calculate_iou_acc(pred_bboxes,gt_bboxes,thresh)
print_ths=[0.5,0.7,0.9]
for k,v in acc.items():
    if(k in print_ths):
        print(f"thresh={k}, acc={v}")
print(f"iou|0.5:0.9={mean([v for v in acc.values()])}")

print(iou)
print(acc)
print(f"{len(iou)}/{len(all_annts)}")

[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
thresh=0.5, acc=0.565693593812866
thresh=0.7, acc=0.35146855022575885
thresh=0.9, acc=0.05279628056685592
iou|0.5:0.9=0.33028099403440275
tensor([0.3408, 0.3189, 0.7189,  ..., 0.7707, 0.8109, 0.7627])
{0.5: 0.565693593812866, 0.55: 0.525973445393178, 0.6: 0.47836291295989986, 0.65: 0.4198220751933479, 0.7: 0.35146855022575885, 0.75: 0.2722070722875408, 0.8: 0.19077741517278377, 0.85: 0.11542760069739372, 0.9: 0.05279628056685592}
44738/44738


In [12]:
for idx,annt in enumerate(all_annts):
    annt['groundingGPT_pred']=dict(
        pred_bboxes=pred_bboxes[idx].tolist(),
        iou=iou[idx].item()
    )


In [14]:
with open('/home/v-jinjzhao/datasets/rec_human_celeb_ready/person_annts_val_ready.json','w') as f:
    json.dump(all_annts,f,indent=4)

In [15]:
num_shikra=0
num_groundGPT=0
for annt in all_annts:
    if('shikra7b_pred' in annt):
        num_shikra+=1
    if('groundingGPT_pred' in annt):
        num_groundGPT+=1
print(f"num_shikra={num_shikra}, num_groundGPT={num_groundGPT}")


num_shikra=41218, num_groundGPT=44738
